In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
import json
import os

from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [3]:
!kaggle datasets download meemr5/indian-names-boys-girls

Dataset URL: https://www.kaggle.com/datasets/meemr5/indian-names-boys-girls
License(s): CC0-1.0
  0% 0.00/161k [00:00<?, ?B/s]
100% 161k/161k [00:00<00:00, 417MB/s]


In [4]:
!unzip indian-names-boys-girls.zip

Archive:  indian-names-boys-girls.zip
  inflating: Names.txt               


In [20]:
names = open("Names.txt", "r").read().lower().splitlines()
names[:5]

['aaban', 'aabharan', 'aabhas', 'aabhat', 'aabheer']

In [21]:
len(names)

55691

In [27]:
names = [name for name in names if '-' not in name and '.' not in name and ' ' not in name]

In [28]:
len(names)

54784

In [29]:
chars = sorted(list(set(''.join(names))))
chars = [char for char in chars if char not in [' ', '-', '.']]
chars

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [30]:
char2idx = {s: i+1 for i,s in enumerate(chars)}
char2idx['.'] = 0
idx2char = {i: s for s,i in char2idx.items()}

In [31]:
char2idx

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [32]:
idx2char

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [33]:
# Building the Dataset
def dataset(names, block_size):
  X, y = [], []
  for name in names:
    context = [0] * block_size
    for ch in name + '.':
      idx = char2idx[ch]
      X.append(context)
      y.append(idx)
      context = context[1:] + [idx]

  X = torch.tensor(X)
  y = torch.tensor(y)
  print(X.shape, y.shape)
  return X, y

In [34]:
dataset(names[:2], 3)

torch.Size([15, 3]) torch.Size([15])


(tensor([[ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1,  1],
         [ 1,  1,  2],
         [ 1,  2,  1],
         [ 2,  1, 14],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1,  1],
         [ 1,  1,  2],
         [ 1,  2,  8],
         [ 2,  8,  1],
         [ 8,  1, 18],
         [ 1, 18,  1],
         [18,  1, 14]]),
 tensor([ 1,  1,  2,  1, 14,  0,  1,  1,  2,  8,  1, 18,  1, 14,  0]))

In [122]:
import random

random.seed(42)
random.shuffle(names)
n1 = int(0.8 * len(names))
n2 = int(0.9 * len(names))

Xtr, ytr = dataset(names[:n1], 3)
Xval, yval = dataset(names[n1:n2], 3)
xte, yte = dataset(names[n2:], 3)

torch.Size([395661, 3]) torch.Size([395661])
torch.Size([49547, 3]) torch.Size([49547])
torch.Size([49439, 3]) torch.Size([49439])


In [123]:
g = torch.Generator().manual_seed(42)
C = torch.randn(27, 3, generator=g)
W1 = torch.randn(9, 100, generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn(100, 27, generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [124]:
def train(X, y, epochs, batch_size):
  for p in parameters:
    p.requires_grad = True

  for _ in range(epochs):
    # Forward Pass
    idx = torch.randint(0, X.shape[0], (batch_size,))
    emb = C[X[idx]]
    h = torch.tanh(emb.view(-1, 9) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y[idx])

    # Backward Pass
    for p in parameters:
      p.grad = None
    loss.backward()

    # Update
    for p in parameters:
      p.data += -1 * p.grad

  print(f'{epochs}: {loss}')

In [125]:
train(Xtr, ytr, 1000, 512)

1000: 2.2689261436462402


In [126]:
def validate(X, y):
  emb = C[X]
  h = torch.tanh(emb.view(-1, 9) @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, y)
  return loss.item()

In [127]:
validate(Xval, yval)

2.226583480834961

In [128]:
validate(Xtr, ytr)

2.230616569519043

In [138]:
def generate_names(block, number):
  out =[]
  for _ in range(number):
    context = [0] * block
    name = ''
    while True:
      emb = C[torch.tensor(context)]
      h = torch.tanh(emb.view(-1, 9) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim = 1)
      idx = torch.multinomial(probs, num_samples=1).item()
      context = context[1:] + [idx]
      if idx == 0:
        break
      name += (idx2char[idx])
    out.append(name)
  return out

In [139]:
generate_names(3, 20)

['yiha',
 'sanishin',
 'sahshissha',
 'densajatharo',
 'meshav',
 'saili',
 'musharshuyani',
 'mura',
 'sayjuci',
 'bahshivoaresunijanirdrangeshin',
 'shinbelvinshiralyan',
 'pamyaari',
 'meshani',
 'tapriya',
 'yaribishuthin',
 'preshini',
 'putt',
 'hith',
 'vath',
 'karuershi']